# ASG Airlines: End-to-End Data Engineering Case Study

## Problem Statement

ASG Airlines, a nationwide carrier, operates flights across multiple cities through its booking platforms, scheduling systems, and airport logs. The organization collects operational flight data from these various systems; however, inconsistencies in the dataset such as corrupted flight identifiers, inconsistent time formats, missing values, and unhandled overnight (cross-day) flight scenarios have created challenges in generating accurate operational insights.

Without a standardized and reliable data pipeline, the airline faces difficulties in calculating accurate flight durations, analyzing route-wise traffic, identifying delays and anomalies, and making data-driven operational decisions.

To improve operational efficiency and reporting accuracy, ASG Airlines aims to build an end-to-end data engineering pipeline that ingests, cleans, standardizes, and transforms flight data into a reliable analytical dataset for reporting and business intelligence purposes.


In [7]:
import pandas as pd

Loading the dataset: (4 separate dataframes since the file has 4 sheets)

In [8]:
flights = pd.read_excel(r"data\UseCase - Airlines.xlsx", sheet_name = "flights") # flights sheet

In [9]:
flights.head()

,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [ ]:
payments = pd.read_excel(r"data\UseCase - Airlines.xlsx", sheet_name = "payments") # payments sheet

In [11]:
payments.head()

,payment_id,booking_id,amount,payment_method
0,PAY1000,B1116,9883.49,NETBANKING
1,PAY1001,B1738,8457.96,NETBANKING
2,PAY1002,B1873,6495.37,UPI
3,PAY1003,B1914,5079.38,NETBANKING
4,PAY1004,B1967,12518.31,CARD


In [12]:
bookings = pd.read_excel(r"data\UseCase - Airlines.xlsx", sheet_name = "bookings") # bookings sheet

In [13]:
bookings.head()

,booking_id,passenger_id,flight_id,booking_date,status,passport_number,seat_number,emergency_contact_name,emergency_contact_phone
0,B1000,P1591,AI192,2025-06-14 11:37:36.951,CANCELLED,P1945887,3D,Isaac Bakshi,+91-6478475128
1,B1001,P1803,6F026,2025-11-02 11:37:36.951,CANCELLED,L3482012,18A,Anvi Konda,+91-6647078662
2,B1002,P1083,SJ010,2025-08-25 11:37:36.951,CANCELLED,G8507659,30C,Udant Dewan,+91-8405938220
3,B1003,P1364,AI069,2025-12-30 11:37:36.951,CONFIRMED,M0891776,33A,Harsh Chahal,+91-6264636839
4,B1004,P1885,UK003,2025-10-02 11:37:36.951,PENDING,N5742231,25C,Pahal Balay,+91-9336478266


In [14]:
passengers = pd.read_excel(r"data\UseCase - Airlines.xlsx", sheet_name = "passengers") # passengers sheet

In [15]:
passengers.head()

,passenger_id,first_name,last_name,age,gender,email,phone,aadhaar_id,date_of_birth
0,P1000,Vivaan,Chatterjee,52,F,vivaan.chatterjee@gmail.com,+91-6896233790,433218196001,1974-04-08
1,P1001,Krishna,Reddy,15,M,krishna.reddy@hotmail.com,+91-6702632297,386379402654,2011-03-07
2,P1002,Myra,Naidu,72,M,myra.naidu@outlook.com,+91-6199585092,615594078161,1954-09-10
3,P1003,Myra,Mishra,61,F,myra.mishra@hotmail.com,+91-8719927151,310341316475,1965-03-12
4,P1004,Saanvi,Banerjee,21,M,saanvi.banerjee@outlook.com,+91-7819595113,419283276483,2005-11-11


Starting EDA for the "flights" dataframe first:

In [16]:
flights.shape

(1020, 7)

In [17]:
flights.dtypes

flight_id                    str
airline                      str
source                       str
destination                  str
departure_time    datetime64[us]
arrival_time      datetime64[us]
duration                  object
dtype: object

In [18]:
flights.isna().sum()

flight_id          0
airline           41
source             0
destination        0
departure_time     0
arrival_time       0
duration           0
dtype: int64

Summary (flights):

- 1020 rows, 7 columns
- duration is being treated as object, can be dropped and calculated using "departure_time" and "arrival_time"
- "airline" has 41 null values (does not account for "UNKNOWN" values, which were observed while scanning the dataset file)